# Project V — Computational Discovery
## Final Scientific Synthesis and Closeout

This notebook reads the existing Project V outputs and creates the final scientific synthesis products. It does not refit clustering models, rerun M2-M4 experiments, or change prior scientific outputs.

The purpose is to separate computational reproducibility, model stability, cross-method recurrence, cross-domain consistency, physical validation, and common-origin interpretation.


In [1]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
FIG = ROOT / 'figures'
REPORT = ROOT / 'report'
FIG.mkdir(exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(exist_ok=True)

OUT_KEY = DATA / 'project_v_final_key_results.csv'
OUT_MATRIX = DATA / 'project_v_final_evidence_matrix.csv'
OUT_FIG = FIG / 'project_v_final_evidence_summary.png'
OUT_REPORT = REPORT / 'project_v_final_scientific_synthesis.md'


## Load Existing Outputs

Inputs are limited to completed Project V products and cross-domain validation tables from Projects II/III/IV/VI. The notebook treats current coverage limitations as part of the result.


In [2]:
def read_csv(name):
    p = DATA / name
    if not p.exists():
        raise FileNotFoundError(p)
    return pd.read_csv(p)

m2_assign = read_csv('project_v_m2_cluster_assignments.csv')
m2_locked = read_csv('project_v_m2_locked_model_summary.csv')
m2_recovery = read_csv('project_v_m2_candidate_recovery_summary.csv')
m3_runs = read_csv('project_v_m3_run_summary.csv')
m3_exp = read_csv('project_v_m3_experiment_summary.csv')
m3_groups = read_csv('project_v_m3_group_stability_summary.csv')
m4_membership = read_csv('project_v_gmm_cross_domain_membership.csv')
m4_coverage = read_csv('project_v_gmm_cross_domain_coverage_summary.csv')
m4_evidence = read_csv('project_v_gmm_cross_domain_evidence_assessment.csv')
cross_method = read_csv('project_v_candidate_cross_method_summary.csv')

m2_assign['known_candidate'] = m2_assign['known_candidate'].astype(bool)
reference = m2_assign['gmm_label'].eq(5)
known = m2_assign['known_candidate']

model_setting_families = ['feature_ablation', 'scaler_sensitivity', 'n_components_sensitivity', 'covariance_sensitivity']
model_setting_runs = m3_runs[m3_runs['experiment_family'].isin(model_setting_families)]

def exp_row(family):
    return m3_exp.loc[m3_exp['experiment_family'].eq(family)].iloc[0]

baseline = exp_row('baseline_reproduction')
random_seed = exp_row('random_seed_stability')
subsample = exp_row('subsample_80pct')
gmm_m2 = m2_recovery.loc[m2_recovery['algorithm'].eq('GMM')].iloc[0]

stable_rows = m3_groups.set_index('stability_group')
recovered_stability = stable_rows.loc['24 M2 reference candidates']
additional_stability = stable_rows.loc['8 M2 new members']
omitted_stability = stable_rows.loc['3 M2 omitted candidates']

recovered = m4_membership[m4_membership['m4_group'].eq('recovered_known_candidate')]
additional = m4_membership[m4_membership['m4_group'].eq('additional_gmm_member')]
omitted = m4_membership[m4_membership['m4_group'].eq('omitted_known_candidate')]
parent = m4_membership[m4_membership['m4_group'].eq('parent_comparison')]

summary = {
    'parent_sample': int(len(m2_assign)),
    'known_candidates': int(known.sum()),
    'gmm_reference_component': int(reference.sum()),
    'recovered_known_candidates': int((reference & known).sum()),
    'additional_members': int((reference & ~known).sum()),
    'omitted_candidates': int((~reference & known).sum()),
    'baseline_candidate_enrichment': float(gmm_m2['best_non_noise_group_enrichment']),
    'baseline_ari': float(baseline['mean_ari_vs_m2_gmm']),
    'baseline_membership_recovery': int(baseline['mean_reference_recall'] * reference.sum()),
    'random_seed_mean_jaccard': float(random_seed['mean_jaccard_overlap']),
    'subsample_80pct_mean_jaccard': float(subsample['mean_jaccard_overlap']),
    'model_setting_mean_jaccard': float(model_setting_runs['jaccard_overlap'].mean()),
    'model_setting_min_jaccard': float(model_setting_runs['jaccard_overlap'].min()),
    'recovered_candidate_mean_selection_frequency': float(recovered_stability['mean_selection_frequency']),
    'additional_member_mean_selection_frequency': float(additional_stability['mean_selection_frequency']),
    'omitted_candidate_mean_selection_frequency': float(omitted_stability['mean_selection_frequency']),
    'recovered_orbit_am_consistent': int(recovered['orbit_am_consistency_label'].eq('consistent').sum()),
    'recovered_orbit_am_covered': int(recovered['has_orbital_validation'].sum()),
    'additional_held_out_covered': int(additional[['has_orbital_validation','has_population_validation','has_chemical_readiness','has_validation_risk']].any(axis=1).sum()),
    'omitted_held_out_covered': int(omitted[['has_orbital_validation','has_population_validation','has_chemical_readiness','has_validation_risk']].any(axis=1).sum()),
}

assert summary['parent_sample'] == 1838
assert summary['known_candidates'] == 27
assert summary['gmm_reference_component'] == 32
assert summary['recovered_known_candidates'] == 24
assert summary['additional_members'] == 8
assert summary['omitted_candidates'] == 3
assert math.isclose(summary['baseline_candidate_enrichment'], 51.05555555555556, rel_tol=1e-9)
assert math.isclose(summary['baseline_ari'], 1.0, rel_tol=0, abs_tol=1e-12)
assert summary['baseline_membership_recovery'] == 32

summary


## Final Key Results Table


In [3]:
key_rows = [
    ('parent_sample', summary['parent_sample'], 'project_v_m2_cluster_assignments.csv', 'Project V discovery parent sample.'),
    ('known_candidates', summary['known_candidates'], 'project_v_m2_cluster_assignments.csv', 'External known-candidate control set.'),
    ('gmm_reference_component_size', summary['gmm_reference_component'], 'project_v_m2_cluster_assignments.csv', 'M2 GMM label 5 reference component.'),
    ('recovered_known_candidates', f"{summary['recovered_known_candidates']} / {summary['known_candidates']}", 'project_v_m2_cluster_assignments.csv', 'Known candidates recovered inside the reference component.'),
    ('additional_gmm_members', summary['additional_members'], 'project_v_m2_cluster_assignments.csv', 'Non-candidate stars inside the reference component.'),
    ('omitted_candidates', summary['omitted_candidates'], 'project_v_m2_cluster_assignments.csv', 'Known candidates outside the reference component.'),
    ('baseline_candidate_enrichment', f"{summary['baseline_candidate_enrichment']:.4f}x", 'project_v_m2_candidate_recovery_summary.csv', 'Candidate fraction enrichment relative to parent sample.'),
    ('locked_baseline_ari', f"{summary['baseline_ari']:.6f}", 'project_v_m3_experiment_summary.csv', 'Exact M2 label reproduction under locked configuration.'),
    ('locked_baseline_membership_recovery', f"{summary['baseline_membership_recovery']} / {summary['gmm_reference_component']}", 'project_v_m3_experiment_summary.csv', 'Exact M2 reference-component recovery.'),
    ('random_seed_mean_jaccard', f"{summary['random_seed_mean_jaccard']:.4f}", 'project_v_m3_experiment_summary.csv', '30 random-seed stability tests.'),
    ('subsample_80pct_mean_jaccard', f"{summary['subsample_80pct_mean_jaccard']:.4f}", 'project_v_m3_experiment_summary.csv', '30 independent 80 percent subsample tests.'),
    ('model_setting_mean_jaccard', f"{summary['model_setting_mean_jaccard']:.3f}", 'project_v_m3_run_summary.csv', 'Feature, scaler, component-count, and covariance variants.'),
    ('model_setting_min_jaccard', f"{summary['model_setting_min_jaccard']:.3f}", 'project_v_m3_run_summary.csv', 'Minimum Jaccard among model-setting variants.'),
    ('recovered_candidate_mean_selection_frequency', f"{summary['recovered_candidate_mean_selection_frequency']:.4f}", 'project_v_m3_group_stability_summary.csv', '24 recovered known candidates.'),
    ('additional_member_mean_selection_frequency', f"{summary['additional_member_mean_selection_frequency']:.4f}", 'project_v_m3_group_stability_summary.csv', '8 additional GMM members.'),
    ('omitted_candidate_mean_selection_frequency', f"{summary['omitted_candidate_mean_selection_frequency']:.4f}", 'project_v_m3_group_stability_summary.csv', '3 omitted known candidates.'),
    ('recovered_candidates_orbit_am_consistent', f"{summary['recovered_orbit_am_consistent']} / {summary['recovered_known_candidates']}", 'project_v_gmm_cross_domain_membership.csv', 'Recovered known candidates with orbit-AM consistency label.'),
    ('additional_member_held_out_coverage', f"{summary['additional_held_out_covered']} / {summary['additional_members']}", 'project_v_gmm_cross_domain_membership.csv', 'Additional members with any Project II/III/IV/VI validation row.'),
    ('omitted_candidate_held_out_coverage', f"{summary['omitted_held_out_covered']} / {summary['omitted_candidates']}", 'project_v_gmm_cross_domain_membership.csv', 'Omitted candidates with any Project II/III/IV/VI validation row.'),
]
key_results = pd.DataFrame(key_rows, columns=['metric', 'value', 'source_file', 'note'])
key_results.to_csv(OUT_KEY, index=False)
key_results


## Final Evidence Matrix


In [4]:
evidence_rows = [
    {
        'target_group': 'full_parent_sample',
        'star_count': summary['parent_sample'],
        'discovery_evidence': 'Defines the 1,838-star Project V discovery-parent sample for advanced clustering.',
        'reproducibility_evidence': 'Input sample reused consistently by M2-M4 products.',
        'stability_evidence': 'Not a candidate structure; used as background comparison.',
        'held_out_evidence': 'No parent-wide Project II/III/IV/VI cross-domain validation table is currently available.',
        'coverage_limitation': 'Held-out orbital/population validation products cover the 27 known candidates only.',
        'interpretation': 'Reference population for enrichment and comparison, not a discovered structure.',
        'confidence_class': 'reference_sample',
        'recommended_next_step': 'Use as denominator for candidate enrichment and future selection-function analysis.',
    },
    {
        'target_group': 'known_candidate_set',
        'star_count': summary['known_candidates'],
        'discovery_evidence': 'Cross-method candidate table combines chemo-kinematic flags, PCA/UMAP embeddings, and DBSCAN noise behavior.',
        'reproducibility_evidence': 'Recovered as the external control set throughout M2-M4.',
        'stability_evidence': 'Subdivides into 24 GMM-recovered candidates and 3 GMM-omitted candidates.',
        'held_out_evidence': 'Project II/III/IV/VI candidate-level products cover all 27 known candidates.',
        'coverage_limitation': 'Candidate set predates GMM validation and is not a confirmed physical population.',
        'interpretation': 'Evidence-ranked follow-up candidate set.',
        'confidence_class': 'followup_candidate_catalogue',
        'recommended_next_step': 'Carry into Project II orbital checks and Project VI uncertainty/literature validation.',
    },
    {
        'target_group': 'recovered_candidate_core',
        'star_count': summary['recovered_known_candidates'],
        'discovery_evidence': '24 known candidates fall in the 32-star GMM reference component.',
        'reproducibility_evidence': 'Recovered exactly under the locked M2 baseline and frequently selected across M3 perturbations.',
        'stability_evidence': f"Mean M3 selection frequency = {summary['recovered_candidate_mean_selection_frequency']:.4f}.",
        'held_out_evidence': f"24/24 have held-out candidate-level coverage; {summary['recovered_orbit_am_consistent']}/24 are orbit-AM consistent.",
        'coverage_limitation': 'Support is partial; chemistry remains Fe/H-limited and detailed abundance validation is absent.',
        'interpretation': 'Partially supported candidate core for follow-up prioritization.',
        'confidence_class': 'partially_supported_candidate_core',
        'recommended_next_step': 'Use as highest-priority comparison core for Project II orbital coherence and Project VI validation.',
    },
    {
        'target_group': 'additional_gmm_members',
        'star_count': summary['additional_members'],
        'discovery_evidence': '8 non-candidate stars share the M2 GMM reference component with the recovered candidate core.',
        'reproducibility_evidence': 'Present in the locked baseline but not strongly stable across all perturbations.',
        'stability_evidence': f"Mean M3 selection frequency = {summary['additional_member_mean_selection_frequency']:.4f}.",
        'held_out_evidence': '0/8 have current Project II orbital, Project III population, Project IV chemical-readiness, or Project VI validation-risk rows.',
        'coverage_limitation': 'No independent held-out orbital/population validation is currently available for these stars.',
        'interpretation': 'Inconclusive; follow-up targets only, not confirmed members of a physical population.',
        'confidence_class': 'inconclusive_followup_targets',
        'recommended_next_step': 'Hand off to Project II for distances, velocities, angular momentum, and baseline orbit diagnostics.',
    },
    {
        'target_group': 'omitted_candidates',
        'star_count': summary['omitted_candidates'],
        'discovery_evidence': '3 known candidates do not fall in the M2 GMM reference component.',
        'reproducibility_evidence': 'Consistently low selection frequency for this specific GMM component.',
        'stability_evidence': f"Mean M3 selection frequency = {summary['omitted_candidate_mean_selection_frequency']:.4f}.",
        'held_out_evidence': '3/3 have held-out candidate-level coverage and retain plausible orbital/population evidence.',
        'coverage_limitation': 'Small n; omission from this GMM component is not evidence against historical candidate status.',
        'interpretation': 'Valid follow-up candidates outside the specific M2 GMM component.',
        'confidence_class': 'candidate_not_selected_by_gmm_component',
        'recommended_next_step': 'Keep in Project II/VI candidate validation; do not treat GMM omission as physical rejection.',
    },
    {
        'target_group': 'full_32_star_reference_component',
        'star_count': summary['gmm_reference_component'],
        'discovery_evidence': 'Nine-component full-covariance GMM identifies a 32-star component with 24/27 known candidates and 51.0556x enrichment.',
        'reproducibility_evidence': 'Locked M2 configuration reproduces ARI = 1.000000 and 32/32 membership recovery.',
        'stability_evidence': f"Random-seed mean Jaccard = {summary['random_seed_mean_jaccard']:.4f}; 80 percent subsample mean Jaccard = {summary['subsample_80pct_mean_jaccard']:.4f}; model-setting mean/min Jaccard = {summary['model_setting_mean_jaccard']:.3f}/{summary['model_setting_min_jaccard']:.3f}.",
        'held_out_evidence': 'Held-out support applies to the 24 recovered known candidates, not to the 8 additional members.',
        'coverage_limitation': 'The additional members have 0/8 held-out coverage; full component is not physically validated.',
        'interpretation': 'Model-dependent computational structure retained as a follow-up prioritization signal.',
        'confidence_class': 'model_dependent_not_physically_validated',
        'recommended_next_step': 'Use as a target list for Project II/VI validation rather than as a physical membership catalogue.',
    },
]
evidence_matrix = pd.DataFrame(evidence_rows)
evidence_matrix.to_csv(OUT_MATRIX, index=False)
evidence_matrix


## Final Evidence Figure

Metrics are split into panels so that recovery counts, Jaccard stability, selection frequencies, and coverage are not treated as one interchangeable scale.


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
fig.suptitle('Project V — Computational Discovery\nFinal Evidence Synthesis', fontsize=16, fontweight='bold')

# Panel A: baseline recovery counts.
ax = axes[0, 0]
labels = ['Known candidates', 'Recovered in GMM', 'Additional members', 'Omitted candidates']
values = [summary['known_candidates'], summary['recovered_known_candidates'], summary['additional_members'], summary['omitted_candidates']]
colors = ['#7b8da8', '#2f6f9f', '#d3922f', '#8f4b4b']
ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_ylabel('Stars')
ax.set_title('Baseline Recovery and Membership')
ax.tick_params(axis='x', rotation=20)
for i, v in enumerate(values):
    ax.text(i, v + 0.6, str(v), ha='center', va='bottom', fontsize=10)
ax.text(0.5, 0.88, f"GMM enrichment = {summary['baseline_candidate_enrichment']:.2f}x", transform=ax.transAxes, ha='center', fontsize=10)

# Panel B: stability families.
ax = axes[0, 1]
stab_labels = ['Locked\nbaseline', 'Random\nseeds', '80%\nsubsamples', 'Model\nsettings']
stab_values = [1.0, summary['random_seed_mean_jaccard'], summary['subsample_80pct_mean_jaccard'], summary['model_setting_mean_jaccard']]
stab_mins = [1.0, float(random_seed['min_jaccard_overlap']), float(subsample['min_jaccard_overlap']), summary['model_setting_min_jaccard']]
ax.bar(stab_labels, stab_values, color=['#3d6fb6', '#5aa469', '#c88a3d', '#8f6bb3'], edgecolor='white', linewidth=0.8)
ax.scatter(range(len(stab_mins)), stab_mins, color='black', s=35, zorder=3, label='Minimum')
ax.set_ylim(0, 1.08)
ax.set_ylabel('Jaccard overlap with 32-star reference')
ax.set_title('Reproducibility and Stability')
ax.legend(frameon=True, fontsize=9)
for i, v in enumerate(stab_values):
    ax.text(i, v + 0.03, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# Panel C: group selection frequencies.
ax = axes[1, 0]
group_labels = ['Recovered\nknown', 'Additional\nGMM', 'Omitted\nknown']
sel_values = [summary['recovered_candidate_mean_selection_frequency'], summary['additional_member_mean_selection_frequency'], summary['omitted_candidate_mean_selection_frequency']]
ax.bar(group_labels, sel_values, color=['#2f6f9f', '#d3922f', '#8f4b4b'], edgecolor='white', linewidth=0.8)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Mean M3 selection frequency')
ax.set_title('Per-Star Stability by Membership Group')
for i, v in enumerate(sel_values):
    ax.text(i, v + 0.035, f'{v:.4f}', ha='center', va='bottom', fontsize=9)

# Panel D: held-out coverage and interpretation.
ax = axes[1, 1]
coverage_labels = ['Recovered\nknown', 'Additional\nGMM', 'Omitted\nknown']
coverage_values = [summary['recovered_orbit_am_covered'] / summary['recovered_known_candidates'], summary['additional_held_out_covered'] / summary['additional_members'], summary['omitted_held_out_covered'] / summary['omitted_candidates']]
ax.bar(coverage_labels, coverage_values, color=['#2f6f9f', '#d3922f', '#8f4b4b'], edgecolor='white', linewidth=0.8)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Fraction with held-out coverage')
ax.set_title('Cross-Domain Coverage')
for i, v in enumerate(coverage_values):
    ax.text(i, v + 0.03, f'{v:.0%}', ha='center', va='bottom', fontsize=10)
ax.text(0.5, -0.22, 'Final classification: model-dependent follow-up signal; not physically validated as a 32-star population.', transform=ax.transAxes, ha='center', va='top', fontsize=9)

for ax in axes.flat:
    ax.spines[['top', 'right']].set_visible(False)

fig.savefig(OUT_FIG, dpi=180, bbox_inches='tight')
plt.show()
print(OUT_FIG.relative_to(ROOT), OUT_FIG.stat().st_size)


<string>:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
figures/project_v_final_evidence_summary.png 252344


## Final Report


In [6]:
report_text = f"""
# Project V — Computational Discovery
## Final Scientific Synthesis and Closeout

Date: 2026-07-27

## Executive Summary

Project V developed the computational-discovery layer of the Gaia-LAMOST Galactic Archaeology research program. It used feature-space design, PCA, UMAP, DBSCAN, HDBSCAN, OPTICS, Gaussian Mixture Models, stability tests, and cross-domain validation audits to ask whether candidate stars recur as coherent computational structures.

The final result is cautious and deliberately bounded. The strongest Project V structure is the M2 GMM 32-star reference component: it contains 24 of the 27 known candidates plus 8 additional stars and has a baseline candidate enrichment of {summary['baseline_candidate_enrichment']:.4f}x. The locked M2 configuration is exactly reproducible in M3, with ARI = {summary['baseline_ari']:.6f} and {summary['baseline_membership_recovery']} / {summary['gmm_reference_component']} reference members recovered.

However, the same component is sensitive to sampling and model specification. Across 30 random-seed runs the mean Jaccard overlap is {summary['random_seed_mean_jaccard']:.4f}; across 30 independent 80 percent subsamples it is {summary['subsample_80pct_mean_jaccard']:.4f}; across feature, scaler, component-count, and covariance variants the mean Jaccard is {summary['model_setting_mean_jaccard']:.3f} and the minimum is {summary['model_setting_min_jaccard']:.3f}. Project V therefore classifies the GMM component as a model-dependent computational structure.

The 24 recovered known candidates form a comparatively stable candidate core with mean selection frequency {summary['recovered_candidate_mean_selection_frequency']:.4f}; {summary['recovered_orbit_am_consistent']} / {summary['recovered_known_candidates']} have Project II orbit-AM consistency. The 8 additional members have lower mean selection frequency ({summary['additional_member_mean_selection_frequency']:.4f}) and 0 / 8 current held-out orbital, population, chemical-readiness, or validation-risk coverage. The 3 omitted candidates have low selection frequency ({summary['omitted_candidate_mean_selection_frequency']:.4f}) for this GMM component, but 3 / 3 retain held-out candidate-level coverage and should not be rejected as candidates merely because they are omitted by this model.

> Computational recovery is not equivalent to physical discovery.

> Cross-domain agreement can strengthen the case for follow-up, but it does not by itself establish a common physical origin.

Project V is therefore closed as a **scientific synthesis completed** computational-discovery project. Its proper output is follow-up prioritization, not a physical discovery claim.

## Research Question

Project V asked whether unsupervised and statistical methods can identify recurring candidate structures in Gaia-LAMOST chemo-kinematic feature space, and whether those structures are reproducible, stable, and scientifically interpretable.

The answer is mixed but useful: several methods recover evidence that the known candidates are unusual relative to the parent sample, and the locked GMM component is exactly reproducible, but full physical validation remains outside the evidence currently available in Project V.

## Data Foundation

Project V builds on a 1,838-star Gaia-LAMOST discovery-parent sample and a 27-star known-candidate control set. The advanced GMM baseline used five features: `feh`, `rv`, `tangential_velocity_kms`, `bp_rp`, and `absolute_g_mag`, with robust scaling and a nine-component full-covariance GMM.

Because [Fe/H], radial velocity, tangential velocity, colour, and absolute magnitude were used directly by the GMM, they are descriptive context in the final interpretation rather than independent validation evidence.

## Completed Computational Methods

Project V completed these repository-backed stages:

- Feature-space design for chemo-kinematic candidate analysis.
- PCA baseline embedding.
- UMAP nonlinear embedding.
- DBSCAN baseline clustering on PCA and UMAP views.
- Small DBSCAN parameter-sweep robustness check.
- Cross-method candidate evidence summary for the 27 known candidates.
- Advanced-clustering readiness and feature audit for the 1,838-star parent sample.
- Blind HDBSCAN, OPTICS, and GMM baseline comparison.
- GMM stability and sensitivity validation.
- Cross-domain validation of the GMM reference component using available Project II/III/IV/VI candidate-level products.
- Final evidence synthesis and project closeout.

No consensus clustering, detailed abundance modelling, uncertainty propagation, or parent-wide orbital validation is claimed as completed in Project V.

## Candidate-Recovery Results

Earlier PCA, UMAP, and DBSCAN products showed that the 27 known candidates often occupy unusual feature-space regions or DBSCAN noise-like regions, especially in the PCA-based view. The cross-method candidate summary remains a candidate-prioritization product rather than a membership catalogue.

The advanced M2 comparison showed that HDBSCAN and OPTICS did not provide a useful candidate-enriched compact group under the locked selection rules. The GMM did: its best candidate-rich group was label 5 with 32 stars, including 24 known candidates and 8 additional stars.

## GMM Baseline Result

The M2 GMM baseline is the central Project V computational result:

- Parent sample: {summary['parent_sample']} stars
- Known candidates: {summary['known_candidates']}
- Reference component: {summary['gmm_reference_component']} stars
- Recovered known candidates: {summary['recovered_known_candidates']} / {summary['known_candidates']}
- Additional GMM members: {summary['additional_members']}
- Omitted known candidates: {summary['omitted_candidates']}
- Baseline candidate enrichment: {summary['baseline_candidate_enrichment']:.4f}x

This is strong evidence that the locked GMM can isolate a candidate-enriched feature-space component. It is not, by itself, evidence that the component is a physically distinct stellar population.

## Reproducibility Assessment

The locked M2 baseline is exactly reproducible in M3:

- ARI versus M2 labels: {summary['baseline_ari']:.6f}
- Reference-component recovery: {summary['baseline_membership_recovery']} / {summary['gmm_reference_component']}

This establishes computational reproducibility for the locked configuration. It does not establish model invariance or physical validity.

## Stability Assessment

Project V M3 shows conditional stability rather than unconditional stability:

- Random-seed mean Jaccard: {summary['random_seed_mean_jaccard']:.4f}
- 80 percent subsampling mean Jaccard: {summary['subsample_80pct_mean_jaccard']:.4f}
- Model-setting mean Jaccard: {summary['model_setting_mean_jaccard']:.3f}
- Model-setting minimum Jaccard: {summary['model_setting_min_jaccard']:.3f}

The recovered known-candidate core is much more stable than the additional or omitted groups:

- 24 recovered candidates: mean selection frequency = {summary['recovered_candidate_mean_selection_frequency']:.4f}
- 8 additional members: mean selection frequency = {summary['additional_member_mean_selection_frequency']:.4f}
- 3 omitted candidates: mean selection frequency = {summary['omitted_candidate_mean_selection_frequency']:.4f}

The GMM reference component is therefore best described as model-dependent. It is robust enough to prioritize follow-up, but not stable enough to define final physical membership.

## Cross-Domain Validation

Project V M4 compared the GMM reference component with available held-out candidate-level products from Project II, Project III, Project IV, and Project VI.

The coverage result is decisive:

- Recovered known candidates: 24 / 24 have candidate-level cross-domain coverage.
- Additional GMM members: 0 / 8 have current held-out coverage.
- Omitted known candidates: 3 / 3 have candidate-level cross-domain coverage.

For the recovered known-candidate core, {summary['recovered_orbit_am_consistent']} / {summary['recovered_known_candidates']} have orbit-AM consistency. This partially supports the candidate core. It does not validate the 8 additional members or the full 32-star component.

Project IV chemistry is currently [Fe/H]-based. Because [Fe/H] is a GMM input feature, it is not independent chemical validation. Detailed abundance information remains a Project VI handoff item.

## Evidence by Membership Group

### 24 Recovered Candidates

The 24 recovered candidates are the strongest Project V output. They are computationally recurrent, have high mean selection frequency, and retain partial held-out orbital support. They should be interpreted as a partially supported candidate core for follow-up prioritization.

### 8 Additional Members

The 8 additional GMM members are not confirmed physical members. They appear in the locked GMM reference component but have lower selection stability and no current held-out orbital, population, chemical-readiness, or validation-risk coverage. They should be treated as follow-up targets only.

### 3 Omitted Candidates

The 3 omitted candidates do not match this specific GMM component and have very low mean selection frequency. That does not invalidate their historical candidate status: all three have held-out candidate-level coverage and plausible orbital/population evidence. They remain Project II/VI validation targets.

### Full 32-Star Reference Component

The full 32-star component is exactly reproducible under the locked M2 configuration and candidate-enriched by construction relative to the parent sample. But because its membership changes under alternative assumptions and because the 8 additional stars lack held-out coverage, it is not physically validated as a common-origin stellar population.

## Interpretation Boundary

Project V distinguishes these concepts:

- Computational reproducibility: the locked M2 GMM can be rerun and recovered exactly.
- Model stability: the component is only conditionally stable under random seeds, subsampling, and model variants.
- Cross-method recurrence: candidate stars recur across PCA, UMAP, DBSCAN, and GMM diagnostics as interesting follow-up targets.
- Cross-domain consistency: existing held-out support applies mainly to the 24 recovered known candidates.
- Physical validation: not complete.
- Common physical origin: not established.

Project V does not claim discovery of a new stellar population, merger remnant, stream, or named Galactic substructure.

## Limitations

The main limitations are:

- Held-out orbital/population validation currently covers the 27 known candidates, not all 1,838 parent stars.
- The 8 additional GMM members lack Project II/III/IV/VI cross-domain rows.
- Current chemistry is limited mostly to [Fe/H] and chemical-readiness metadata.
- GMM membership depends on feature choice, scaling, component count, covariance type, and sampling.
- Selection functions and catalogue completeness have not yet been modelled.
- Orbit uncertainties and Galactic-potential sensitivity remain to be propagated.

## Project-Level Conclusion

Project V is complete as a computational-discovery synthesis. It produced a reproducible, evidence-ranked follow-up framework and identified a model-dependent 32-star GMM component with a stable 24-star known-candidate core.

The final scientific classification is:

- 24 recovered candidates: **partially supported candidate core**.
- 8 additional members: **inconclusive; follow-up targets only**.
- Full 32-star component: **model-dependent and not physically validated**.

The appropriate Project V conclusion is follow-up prioritization, not physical discovery.

## Handoff to Project II and Project VI

### Handoff to Project II — Orbital Dynamics

Project II should extend orbital validation to the 8 additional GMM members by obtaining or deriving reliable distance, position, and velocity information where possible. It should then compute or validate quantities such as `Lz`, `Lperp`, `Ltot`, eccentricity, and `Zmax`, and test whether the additional members share orbital coherence with the recovered-candidate core.

### Handoff to Project VI — Scientific Validation

Project VI should handle uncertainty propagation, selection-function analysis, detailed abundance validation, external catalogue and literature comparison, and physical-origin assessment. It should decide whether any subset of the Project V targets can support a stronger astrophysical claim after uncertainty and external validation.

## Final Deliverables

Notebook:

- `notebooks/29_project_v_final_scientific_synthesis.ipynb`

Processed outputs:

- `data/processed/project_v_final_key_results.csv`
- `data/processed/project_v_final_evidence_matrix.csv`

Figure:

- `figures/project_v_final_evidence_summary.png`

Report:

- `report/project_v_final_scientific_synthesis.md`
""".strip() + "\n"
OUT_REPORT.write_text(report_text)
print(OUT_REPORT.relative_to(ROOT), OUT_REPORT.stat().st_size)


report/project_v_final_scientific_synthesis.md 11553


## Output Check


In [7]:
for path in [OUT_KEY, OUT_MATRIX, OUT_FIG, OUT_REPORT]:
    print(path.relative_to(ROOT), path.stat().st_size)
print(json.dumps(summary, indent=2))


data/processed/project_v_final_key_results.csv 2308
data/processed/project_v_final_evidence_matrix.csv 3978
figures/project_v_final_evidence_summary.png 252344
report/project_v_final_scientific_synthesis.md 11553
{
  "parent_sample": 1838,
  "known_candidates": 27,
  "gmm_reference_component": 32,
  "recovered_known_candidates": 24,
  "additional_members": 8,
  "omitted_candidates": 3,
  "baseline_candidate_enrichment": 51.05555555555555,
  "baseline_ari": 1.0,
  "baseline_membership_recovery": 32,
  "random_seed_mean_jaccard": 0.7479188726441048,
  "subsample_80pct_mean_jaccard": 0.6555275339237506,
  "model_setting_mean_jaccard": 0.7147269295930718,
  "model_setting_min_jaccard": 0.34375,
  "recovered_candidate_mean_selection_frequency": 0.8785569105691057,
  "additional_member_mean_selection_frequency": 0.4131097560975609,
  "omitted_candidate_mean_selection_frequency": 0.056910569105691,
  "recovered_orbit_am_consistent": 21,
  "recovered_orbit_am_covered": 24,
  "additional_held_o